<a href="https://colab.research.google.com/github/Prktriesitout/Langchain_Projects/blob/main/Project3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**AI Financial Report Analyst**

Business Scenario
A financial advisory company regularly receives quarterly and annual reports (PDFs) from multiple firms. Analysts spend significant time reviewing these documents to extract key insights such as revenue trends, expenses, risks, and future outlooks.

Clients often ask multi-turn, contextual questions, for example:

“What was the revenue growth this quarter compared to last?”
“What risks did the company highlight?”
“Can you summarize all risks and growth strategies we discussed earlier?”
Currently, this process is time-consuming and inconsistent across analysts.

The company wants an AI-powered Financial Report Analyst that can automatically process financial documents, summarize insights, and remember prior client questions for more natural and effective interactions.

Develop an AI Financial Report Analyst using LangChain that can:

Load and process financial report PDF>>financial_report.pdf.
Use Prompt Templates to extract structured insights on Revenue, Expenses, Risks, and Outlook.
Enable multi-turn conversations with memory, so the system recalls prior questions and answers.
Provide consistent, finance-domain responses tailored to client queries

Step 1: Setup

Install required libraries: langchain, transformers, huggingface_hub, pypdf.
Import key modules: PyPDFLoader, PromptTemplate, InMemoryChatMessageHistory, LangChain's Chain component.

In [1]:
!pip install langchain-classic

In [2]:
from langchain_classic.chains import RetrievalQA
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import BaseChatMessageHistory, InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint


In [3]:
import os
from google.colab import userdata

# --------------------------
# Setup COHERE API Key from Colab Secrets
# --------------------------
os.environ["COHERE_API_KEY"] = userdata.get("cohere")


In [4]:
!pip install -U langchain-cohere

In [5]:
from langchain_cohere import ChatCohere

# --------------------------
# Import Cohere LLM
# --------------------------
llm = ChatCohere(
    model="command-a-03-2025",  # Cohere's flagship model (can also use "command-r-plus-08-2024" or "command-r-08-2024")
    temperature=0.1,
    max_tokens=300,
)


In [6]:
from langchain_core.chat_history import InMemoryChatMessageHistory

# Dictionary to store session-based message histories
store = {}

# Function to retrieve or create chat history for a given session ID
def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]


Step 2: Load and Ingest Financial Reports

Load the PDF report using PyPDFLoader.
Extract text from the document.
Optionally split text into chunks using RecursiveCharacterTextSplitter.
For demonstration, use a limited set of pages.

Step 3: Define Prompt Template

Create a structured prompt that instructs the AI to:

Identify Revenue Trends.
Summarize Expenses.
List Risks.
Highlight Future Outlook/Strategies.
Ensure outputs are concise and presented in bullet-style format.

In [7]:
!pip install -U langchain-community

In [8]:
!pip install -U langchain-community langchain-text-splitters pypdf

In [9]:
# --------------------------
# Document ingestion
# --------------------------
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyPDFLoader(r"financial_report.pdf")   # Change to your PDF path
documents = loader.load()

# Split into chunks for better retrieval
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    separators=["\n\n", "\n", ".", " ", ""]
)
texts = text_splitter.split_documents(documents)
texts



/tmp/ipykernel_9715/3681741683.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2025-09-09T04:12:40+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2025-09-09T04:12:40+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'financial_report.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='Saudi Aramco Financial Report\nCompany Snapshot\nSaudi Aramco (Saudi Arabian Oil Company) is the world’s largest integrated oil and gas company,\nheadquartered in Dhahran, Saudi Arabia. The company is majority-owned by the Government of\nSaudi Arabia. CEO: Amin H. Nasser. Operations include exploration, production, refining,\ndistribution, and petrochemicals.\nFY 2024 Financial Highlights\nMetric\nValue (USD Billion)\nRevenue\n480.45\nOperating Income\n206.57\nNet Income\n106.25\nTotal Assets\n646.30\nShareholder Equity\n440.36\nDividend Performance\nIn 2024, Aramco declared total dividend pa

In [10]:
# --------------------------
# Embeddings & Vector Store
# --------------------------
from langchain_huggingface import HuggingFaceEmbeddings
model_name = "sentence-transformers/all-mpnet-base-v2"
embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': False}
)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Step 4: Add Conversation Memory

Use InMemoryChatMessageHistory to store prior user questions and responses.

Example flow:

User: “Summarize the revenue growth trend.”
User: “Now tell me about the risks mentioned.”
User: “Combine both insights into a single summary.”
The AI should recall earlier responses and merge them into a consolidated answer.

In [11]:
!pip install -U langchain-chroma

In [12]:
from langchain_chroma import Chroma

# Create Chroma index
db = Chroma.from_documents(texts, embeddings)


In [13]:
retriever=db.as_retriever(search_type="similarity", search_kwargs={"k": 3})


Step 5: Build the LLM Chain

Connect:

HuggingFace LLM / Bedrock LLMs (for text generation).
The Prompt Template (for structured instructions).
Conversation Memory (for context persistence).
Use LangChain's create_stuff_documents_chain and create_retrieval_chain components to link all components.

In [14]:
chat_prompt = ChatPromptTemplate.from_messages([
    # --- System Message ---
    ("system", """
You are a financial report analyst.
Use the retrieved context to answer queries.
If answer for the query is not rooted in the context provided,
answer based on the query appropriately.
Answer any user questions based solely on the context below:
<context>
{context}
</context>
"""),
    # --- Chat History Placeholder ---
    ("placeholder", "{chat_history}"),
    # --- Human Message ---
    ("human", "{input}")
])
#print(chat_prompt)


Step 6: Simulate Client Queries

Demonstrate the system with sample queries:

“What is the company’s revenue growth this year?” → Extracted from the financial report.
“What risks were mentioned?” → AI summarizes reported risks.
“Give me a combined summary of revenue and risks so far.” → AI recalls earlier answers and provides a consolidated summary.

In [15]:
!pip install -U langchain-classic rich


In [18]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from rich import print

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Assumes chat_prompt uses {context} and {input} variables
retriever_chain = (
    {"context": retriever | format_docs, "input": RunnablePassthrough()}
    | chat_prompt
    | llm
    | StrOutputParser()
)


In [19]:
store = {}
def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]


In [20]:
from langchain_core.runnables.history import RunnableWithMessageHistory

chain_with_memory = RunnableWithMessageHistory(
    retriever_chain,
    get_session_history,
    input_messages_key="input",         # Matches {input} in your ChatPromptTemplate
    history_messages_key="chat_history", # Matches {chat_history} in your ChatPromptTemplate
    output_messages_key="answer"         # Matches the output key from create_retrieval_chain
)


/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3553: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [21]:
# --------------------------
# Run queries with session memory
# --------------------------
session_id = "Oct 7"
queries = [
    "What is the company's revenue growth this year 2025?",
    "What risks were mentioned?",
    "Give me a combined summary of revenue and risks so far.",
    "What are my previous queries?"
]


In [24]:
from operator import itemgetter
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain

docs_chain = create_stuff_documents_chain(llm, chat_prompt)

# Extract only the "input" string before sending to the retriever
retriever_chain = create_retrieval_chain(
    itemgetter("input") | retriever,
    docs_chain
)



In [29]:
from operator import itemgetter
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain
from langchain_core.runnables.history import RunnableWithMessageHistory

# 1. Create document chain
docs_chain = create_stuff_documents_chain(llm, chat_prompt)

# 2. Extract only the string from {"input": q} before passing to Chroma's retriever
retriever_chain = create_retrieval_chain(
    itemgetter("input") | retriever,
    docs_chain
)

# 3. Re-wrap the UPDATED retriever_chain with memory
chain_with_memory = RunnableWithMessageHistory(
    retriever_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    output_messages_key="answer"
)

# 4. Run your queries
for i, q in enumerate(queries, 1):
    response = chain_with_memory.invoke(
        {"input": q},
        config={"configurable": {"session_id": session_id}}
    )
    print(f"\nQuery {i}: {q}")
    print("Answer:", response["answer"])



/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3553: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


Query 1: What is the total revenue in the report?

Answer: The total revenue for Saudi Aramco in FY 2024 is **USD 480.45 billion**, as stated in the "FY 2024 
Financial Highlights" section of the context.

Query 2: How does that compare to the previous year?

Answer: The context provided does not include the revenue figure for the year prior to FY 2024, so I cannot provide
a comparison to the previous year based on the information given. If you have additional data or context, I'd be 
happy to assist with the comparison.

Query 3: What are the main risks highlighted?

Answer: The main risks highlighted in the context are:

1. **Sustained oil price volatility**: This is a key challenge for Aramco, as it directly impacts the company's 
revenue and profitability. The significant drop in average realized oil prices from USD 85.7/barrel in Q2 2024 to 
USD 66.7/barrel in Q2 2025 illustrates this risk.

2. **Dividend sustainability pressures**: The company faces challenges in maintaining its dividend payouts, 
particularly performance-linked dividends, which dropped by 98% in H1 2025 due to lower free cash flow resulting 
from weaker oil prices.

3. **Lower free cash flow**: The decline in free cash flow, as seen in the detailed financial analysis (e.g., USD 
15.2 billion in Q2 2025), is a concern, as it affects the company's ability to fund capital expenditures, pay 
dividends, and maintain financial flexibility.

These risks are interconnected, with oil price volatility being the primary driver of the challenges related to 
dividend sustainability and free cash flow.

In [30]:
# --------------------------
# Inspect memory
# --------------------------
from rich import print
print("\nConversation memory messages:")
for m in store[session_id].messages:
    print(f"- {m.type}: {m.content}")


Conversation memory messages:

- human: What is the total revenue in the report?

- ai: The total revenue for Saudi Aramco in FY 2024 is **USD 480.45 billion**, as stated in the "FY 2024 Financial 
Highlights" section of the context.

- human: How does that compare to the previous year?

- ai: The context provided does not include the revenue figure for the year prior to FY 2024, so I cannot provide a
comparison to the previous year based on the information given. If you have additional data or context, I'd be 
happy to assist with the comparison.

- human: What are the main risks highlighted?

- ai: The main risks highlighted in the context are:

1. **Sustained oil price volatility**: This is a key challenge for Aramco, as it directly impacts the company's 
revenue and profitability. The significant drop in average realized oil prices from USD 85.7/barrel in Q2 2024 to 
USD 66.7/barrel in Q2 2025 illustrates this risk.

2. **Dividend sustainability pressures**: The company faces challenges in maintaining its dividend payouts, 
particularly performance-linked dividends, which dropped by 98% in H1 2025 due to lower free cash flow resulting 
from weaker oil prices.

3. **Lower free cash flow**: The decline in free cash flow, as seen in the detailed financial analysis (e.g., USD 
15.2 billion in Q2 2025), is a concern, as it affects the company's ability to fund capital expenditures, pay 
dividends, and maintain financial flexibility.

These risks are interconnected, with oil price volatility being the primary driver of the challenges related to 
dividend sustainability and free cash flow.

In [32]:
import os
import uuid
from operator import itemgetter

import gradio as gr
from google.colab import userdata
from langchain_chroma import Chroma
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_cohere import ChatCohere
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ---------- Setup (loaded once) ----------
os.environ["COHERE_API_KEY"] = userdata.get("cohere")

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": False},
)

llm = ChatCohere(model="command-a-03-2025", temperature=0.1, max_tokens=500)

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    separators=["\n\n", "\n", ".", " ", ""],
)

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are a financial report analyst.
Use the retrieved context to answer queries.
If answer for the query is not rooted in the context provided,
answer based on the query appropriately.
Answer any user questions based solely on the context below:
<context>
{context}
</context>
"""),
    ("placeholder", "{chat_history}"),
    ("human", "{input}"),
])

# ---------- Session storage ----------
CHAINS = {}   # session_id -> RAG chain with memory
HISTORY = {}  # session_id -> chat message history


def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in HISTORY:
        HISTORY[session_id] = InMemoryChatMessageHistory()
    return HISTORY[session_id]


def build_chain(pdf_path):
    pages = PyPDFLoader(pdf_path).load()
    texts = splitter.split_documents(pages)
    if not texts:
        raise ValueError("No text could be extracted (is the PDF scanned?).")

    db = Chroma.from_documents(
        texts, embeddings, collection_name=f"pdf_{uuid.uuid4().hex}"
    )
    retriever = db.as_retriever(search_type="similarity", search_kwargs={"k": 3})

    docs_chain = create_stuff_documents_chain(llm, chat_prompt)
    rag_chain = create_retrieval_chain(itemgetter("input") | retriever, docs_chain)

    chain = RunnableWithMessageHistory(
        rag_chain,
        get_session_history,
        input_messages_key="input",
        history_messages_key="chat_history",
        output_messages_key="answer",
    )
    return chain, len(pages), len(texts)


# ---------- Gradio handlers ----------
def process_pdf(pdf_path):
    if pdf_path is None:
        return None, [], "⚠️ Please upload a PDF."
    try:
        chain, n_pages, n_chunks = build_chain(pdf_path)
        sid = uuid.uuid4().hex
        CHAINS[sid] = chain
        return sid, [], f"✅ Ready: {n_pages} pages, {n_chunks} chunks indexed. Ask away!"
    except Exception as e:
        return None, [], f"❌ Error: {e}"


def respond(message, history, sid):
    if not message or not message.strip():
        return "", history
    history = history + [{"role": "user", "content": message}]

    if sid is None or sid not in CHAINS:
        history.append({"role": "assistant", "content": "⚠️ Please upload and process a PDF first."})
        return "", history

    try:
        result = CHAINS[sid].invoke(
            {"input": message},
            config={"configurable": {"session_id": sid}},
        )
        answer = result["answer"]
        pages = sorted({d.metadata.get("page", 0) + 1 for d in result.get("context", [])})
        if pages:
            answer += f"\n\n<sub>📎 Sources: page {', '.join(map(str, pages))}</sub>"
    except Exception as e:
        answer = f"❌ Error: {e}"

    history.append({"role": "assistant", "content": answer})
    return "", history


def clear_chat(sid):
    HISTORY.pop(sid, None)  # reset memory, keep the indexed PDF
    return [], ""


# ---------- UI ----------
with gr.Blocks(theme=gr.themes.Soft(), title="Financial Report Chat") as demo:
    session_id = gr.State(None)

    gr.Markdown(
        "# 📊 Financial Report Assistant\n"
        "Upload a PDF, then ask questions about it. Follow-up questions are remembered."
    )

    with gr.Row():
        with gr.Column(scale=1):
            pdf_input = gr.File(label="Upload PDF", file_types=[".pdf"], type="filepath")
            process_btn = gr.Button("Process PDF", variant="primary")
            status = gr.Markdown("Upload a PDF to begin.")

        with gr.Column(scale=2):
            chatbot = gr.Chatbot(type="messages", height=450, label="Conversation")
            msg = gr.Textbox(
                placeholder="e.g. What is the total revenue in the report?",
                show_label=False,
            )
            with gr.Row():
                send_btn = gr.Button("Send", variant="primary")
                clear_btn = gr.Button("Clear chat")

    process_btn.click(process_pdf, pdf_input, [session_id, chatbot, status])
    msg.submit(respond, [msg, chatbot, session_id], [msg, chatbot])
    send_btn.click(respond, [msg, chatbot, session_id], [msg, chatbot])
    clear_btn.click(clear_chat, session_id, [chatbot, msg])

demo.launch(debug=True)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

/tmp/ipykernel_9715/995184518.py:129: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="Financial Report Chat") as demo:
/tmp/ipykernel_9715/995184518.py:144: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(type="messages", height=450, label="Conversation")


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://1d67e34177597fea3f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/tmp/ipykernel_9715/995184518.py:90: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  chain, n_pages, n_chunks = build_chain(pdf_path)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://1d67e34177597fea3f.gradio.live


In [ ]:
import os
import uuid

import gradio as gr
from google.colab import userdata
from langchain_chroma import Chroma
from langchain_classic.chains import create_history_aware_retriever, create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_cohere import ChatCohere
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ---------- Setup (loaded once) ----------
os.environ["COHERE_API_KEY"] = userdata.get("cohere")

NOT_FOUND = "I couldn't find that in the uploaded document."

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

llm = ChatCohere(model="command-a-03-2025", temperature=0, max_tokens=500)

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    separators=["\n\n", "\n", ".", " ", ""],
)

# Rewrites follow-ups ("How does that compare to last year?") into standalone
# questions so the retriever searches for the right thing.
contextualize_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Given the chat history and the latest user question, rewrite the question "
     "as a standalone question that can be understood without the history. "
     "Do NOT answer it. If it is already standalone, return it unchanged."),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

# Strict, document-only answering prompt.
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", f"""
You are a financial report analyst.

Answer ONLY using the information inside the <context> block below.
Rules:
1. Do NOT use outside knowledge, assumptions, or general information.
2. If the context does not contain the answer, reply exactly with:
   "{NOT_FOUND}"
3. Quote figures exactly as written in the context, including units and periods.
4. Be concise. Use bullet points for lists of items.

<context>
{{context}}
</context>
"""),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

# ---------- Session storage ----------
CHAINS = {}
HISTORY = {}


def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in HISTORY:
        HISTORY[session_id] = InMemoryChatMessageHistory()
    return HISTORY[session_id]


def build_chain(pdf_path, k):
    pages = PyPDFLoader(pdf_path).load()
    texts = splitter.split_documents(pages)
    if not texts:
        raise ValueError("No text could be extracted (is the PDF scanned?).")

    db = Chroma.from_documents(
        texts,
        embeddings,
        collection_name=f"pdf_{uuid.uuid4().hex}",
        collection_metadata={"hnsw:space": "cosine"},
    )
    retriever = db.as_retriever(search_type="similarity", search_kwargs={"k": int(k)})

    history_aware = create_history_aware_retriever(llm, retriever, contextualize_prompt)
    docs_chain = create_stuff_documents_chain(llm, qa_prompt)
    rag_chain = create_retrieval_chain(history_aware, docs_chain)

    chain = RunnableWithMessageHistory(
        rag_chain,
        get_session_history,
        input_messages_key="input",
        history_messages_key="chat_history",
        output_messages_key="answer",
    )
    return chain, len(pages), len(texts)


# ---------- Gradio handlers ----------
def process_pdf(pdf_path, k):
    if pdf_path is None:
        return None, [], "⚠️ Please upload a PDF."
    try:
        chain, n_pages, n_chunks = build_chain(pdf_path, k)
        sid = uuid.uuid4().hex
        CHAINS[sid] = chain
        return sid, [], f"✅ Ready: {n_pages} pages, {n_chunks} chunks indexed. Ask away!"
    except Exception as e:
        return None, [], f"❌ Error: {e}"


def respond(message, history, sid):
    if not message or not message.strip():
        return "", history
    history = history + [{"role": "user", "content": message}]

    if sid is None or sid not in CHAINS:
        history.append({"role": "assistant", "content": "⚠️ Please upload and process a PDF first."})
        return "", history

    try:
        result = CHAINS[sid].invoke(
            {"input": message},
            config={"configurable": {"session_id": sid}},
        )
        answer = result["answer"].strip()

        # Only show sources when the answer actually came from the document
        if NOT_FOUND not in answer:
            pages = sorted({d.metadata.get("page", 0) + 1 for d in result.get("context", [])})
            if pages:
                answer += f"\n\n<sub>📎 Sources: page {', '.join(map(str, pages))}</sub>"
    except Exception as e:
        answer = f"❌ Error: {e}"

    history.append({"role": "assistant", "content": answer})
    return "", history


def clear_chat(sid):
    HISTORY.pop(sid, None)
    return [], ""


# ---------- UI ----------
with gr.Blocks(theme=gr.themes.Soft(), title="Financial Report Chat") as demo:
    session_id = gr.State(None)

    gr.Markdown(
        "# 📊 Financial Report Assistant\n"
        "Answers come **only from your uploaded document**. Follow-up questions are remembered."
    )

    with gr.Row():
        with gr.Column(scale=1):
            pdf_input = gr.File(label="Upload PDF", file_types=[".pdf"], type="filepath")
            top_k = gr.Slider(
                2, 8, value=4, step=1,
                label="Passages retrieved per question",
                info="Higher = more context, slightly slower.",
            )
            process_btn = gr.Button("Process PDF", variant="primary")
            status = gr.Markdown("Upload a PDF to begin.")

        with gr.Column(scale=2):
            chatbot = gr.Chatbot(type="messages", height=450, label="Conversation")
            msg = gr.Textbox(
                placeholder="e.g. What is the total revenue in the report?",
                show_label=False,
            )
            with gr.Row():
                send_btn = gr.Button("Send", variant="primary")
                clear_btn = gr.Button("Clear chat")

    process_btn.click(process_pdf, [pdf_input, top_k], [session_id, chatbot, status])
    msg.submit(respond, [msg, chatbot, session_id], [msg, chatbot])
    send_btn.click(respond, [msg, chatbot, session_id], [msg, chatbot])
    clear_btn.click(clear_chat, session_id, [chatbot, msg])

demo.launch(debug=True)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

/tmp/ipykernel_9715/3207627059.py:154: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="Financial Report Chat") as demo:
/tmp/ipykernel_9715/3207627059.py:174: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(type="messages", height=450, label="Conversation")


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://c27aa7792a63414cc5.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 422, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        self.scope, self.receive, self.send
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/uvicorn/middleware/proxy_headers.py", line 63, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/fastapi/applications.py", line 1163, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.13/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.13/dist-packages/starlette/middleware/errors.py", line 186, 